# PA-RL distillation: ResNet critic → SmolVLA

**Setup по статье PA-RL (Algorithm 1):**
1. Загружаем уже обученный ResNet critic (`q_demo - q_random ~17`)
2. Загружаем SmolVLA baseline
3. На каждом training step:
   - Sample N кандидатов через flow-matching (denoise 4 steps)
   - Global optimization: re-rank по Q, top-m
   - Local optimization: M grad steps на action
   - Distillation: BC loss policy на выбранный a* через flow-matching forward
4. Оценка на 13 init_states

**Адаптации к нашему setup:**
- Critic работает на raw images 84×84 + state, не VLM embeddings (и так и было задумано)
- Distillation **только на первой позиции chunk'а** (`losses[:, 0, :7]`) — критическое исправление прошлого debug сеанса
- Frozen всё кроме action_expert — radikally снижает overfit risk
- LR=1e-5, мало шагов (200) — против mode collapse

**Гиперпараметры по статье (Appendix B.1, real robot):**
- N_CANDIDATES=8 (мы видели — больше не помогает)
- M_GLOBAL=2 (top-m кандидатов берём для local refine)
- N_REFINEMENT_STEPS=10
- η_local=3e-4
- CAL_ALPHA=0.01 (для PA-RL phase)

## 1. Setup, конфиг, загрузка критика и SmolVLA

In [ ]:
import os
os.environ["MUJOCO_GL"]="egl"; os.environ["PYOPENGL_PLATFORM"]="egl"

import sys, math, time, gc, copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

print(f"PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")
device = torch.device("cuda:0")
torch.backends.cudnn.benchmark = True

MODEL_ID            = "HuggingFaceVLA/smolvla_libero"
CRITIC_PATH         = "/workspace/out/critic_resnet.pt"
SAVE_VLA_PATH       = "/workspace/out/smolvla_parl_resnet"

TASK_DESCRIPTION = "pick up the black bowl between the plate and the ramekin and place it on the plate"

TASK_SUITE_NAME       = "libero_spatial"
TASK_ID               = 0
ENV_IMAGE_SIZE        = 256
SEED                  = 42
MAX_STEPS             = 90
NUM_STABILIZATION_STEPS = 10
INIT_STATES_IDS       = [1, 2, 6, 7, 13, 22, 23, 27, 32, 35, 38, 47, 46]  # canonical eval set — не менять! (соответствует 01_data_prep)

# === PA-RL hyperparameters (по статье + sweet spot из SFT progressive) ===
# SFT progressive показал: SR улучшается на step 20-60, потом резко обваливается на 80.
# Sweet spot для distillation flow-matching = 20-60 шагов.
EVAL_AT_STEPS      = [0, 20, 40, 60]  # точки эвала
PARL_STEPS         = max(EVAL_AT_STEPS)  # 60
BATCH_SIZE         = 64
N_CANDIDATES       = 16  # было 8: paper default 32, для OpenVLA real-robot 16
M_GLOBAL           = 4   # было 2: с k=16 берём 25% (paper App B.1: top-10 из 32)
N_REFINEMENT_STEPS = 10
ETA_LOCAL          = 3e-4
NUM_DENOISE_STEPS  = 10
CAL_ALPHA          = 0.01
POLICY_LR          = 3e-6   # было 1e-5: flow-matching SmolVLA нестабилен на больших LR
LOG_EVERY          = 10

torch.manual_seed(SEED); np.random.seed(SEED)

from huggingface_hub import login
login(token="", add_to_git_credential=False)
print("[hf] logged in")


## 2. Загрузка SmolVLA

In [ ]:
from lerobot.policies.smolvla.modeling_smolvla import (
    SmolVLAPolicy, resize_with_pad, pad_vector, make_att_2d_masks
)
from lerobot.policies.factory import make_pre_post_processors

policy = SmolVLAPolicy.from_pretrained(MODEL_ID).to(device)
preprocessor, postprocessor = make_pre_post_processors(
    policy_cfg=policy.config, pretrained_path=MODEL_ID,
    preprocessor_overrides={"device_processor": {"device": str(device)}},
)
print(f"Policy loaded, chunk_size={policy.config.chunk_size}, num_steps={policy.config.num_steps}")

# ─── LoRA fine-tuning по статье PA-RL (rank=32 для OpenVLA) ───
# Цитата из статьи:
#   "we distill optimized actions into OpenVLA via LoRA fine-tuning with rank=32"
# 
# Полное unfreezing action_expert ломает flow-matching policy (видели mode collapse).
# LoRA добавляет малую adaptive модификацию ВНУТРИ слоёв, основные веса остаются нетронутыми.

n_total = sum(p.numel() for p in policy.parameters())

# Freeze ВСЕ параметры
for p in policy.parameters():
    p.requires_grad_(False)

# Применим LoRA вручную (без peft library — кастомный wrapping)
class LoRALinear(nn.Module):
    """Low-rank adaptation на nn.Linear: out = base(x) + (alpha/r) * B(A(x)).
    
    Прозрачный wrapper: внешне ведёт себя как nn.Linear (weight, bias, in_features,
    out_features доступны), внутри делегирует forward к baseline + LoRA delta.
    Это нужно потому что внутренний код LeRobot читает .weight.dtype напрямую."""
    def __init__(self, base_layer: nn.Linear, rank: int = 32, alpha: float = 32.0):
        super().__init__()
        self.base = base_layer
        for p in self.base.parameters():
            p.requires_grad_(False)
        in_f = base_layer.in_features
        out_f = base_layer.out_features
        self.in_features = in_f
        self.out_features = out_f
        self.lora_A = nn.Linear(in_f, rank, bias=False)
        self.lora_B = nn.Linear(rank, out_f, bias=False)
        # init: A — Kaiming uniform, B — zeros (так чтобы старт был identity к baseline)
        nn.init.kaiming_uniform_(self.lora_A.weight, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B.weight)
        self.scaling = alpha / rank
    
    @property
    def weight(self):
        # Для обратной совместимости — внешний код может читать .weight.dtype/.device
        return self.base.weight
    
    @property
    def bias(self):
        return self.base.bias
    
    def forward(self, x):
        # Сохраняем dtype baseline path — LoRA path подгоняется
        base_out = self.base(x)
        # x может быть в другом dtype чем lora_A.weight
        x_lora = x.to(self.lora_A.weight.dtype)
        lora_out = self.scaling * self.lora_B(self.lora_A(x_lora))
        return base_out + lora_out.to(base_out.dtype)


def apply_lora(module, target_names, rank=32, alpha=32):
    """Рекурсивно заменяет nn.Linear на LoRALinear для слоёв чьи имена содержат target_names."""
    n_replaced = 0
    for name, child in list(module.named_children()):
        if isinstance(child, nn.Linear) and any(t in name for t in target_names):
            setattr(module, name, LoRALinear(child, rank=rank, alpha=alpha))
            n_replaced += 1
        else:
            n_replaced += apply_lora(child, target_names, rank, alpha)
    return n_replaced


# Применяем LoRA к Q/K/V проекциям и выходным проекциям action_expert.
# Имена зависят от внутреннего LeRobot — берём общие шаблоны.
LORA_RANK = 32
LORA_ALPHA = 8   # было 32: scaling=alpha/rank=0.25 для замедления коллапса

# action_expert это policy.model.vlm_with_expert.lm_expert
target_names = ["q_proj", "k_proj", "v_proj", "o_proj"]  # стандартные attention слои

if hasattr(policy.model.vlm_with_expert, "lm_expert"):
    n_lora = apply_lora(policy.model.vlm_with_expert.lm_expert, target_names,
                        rank=LORA_RANK, alpha=LORA_ALPHA)
    print(f"Applied LoRA to {n_lora} attention projections in lm_expert")
else:
    print("WARN: lm_expert не найден в expected месте")

# ── Только LoRA, как в paper Section 5.2.2 для OpenVLA ──
# Раньше был unfreeze action_out_proj — этот слой не регуляризован LoRA-структурой
# и может уезжать произвольно. Paper-faithful: только LoRA.

# Перенести LoRA параметры на device
policy.to(device)

n_trainable = sum(p.numel() for p in policy.parameters() if p.requires_grad)
print(f"Total params: {n_total/1e6:.1f}M")
print(f"Trainable (LoRA only): {n_trainable/1e6:.2f}M ({100*n_trainable/n_total:.2f}%)")

# ── Согласовать NUM_DENOISE_STEPS с тем, что используется в eval ──
# В eval policy.select_action использует policy.config.num_steps. Если sampling
# в PA-RL делает меньше шагов, кандидаты низкого качества → distill target плохой.
NUM_DENOISE_STEPS = policy.config.num_steps
print(f"NUM_DENOISE_STEPS overridden to policy.config.num_steps = {NUM_DENOISE_STEPS}")


## 3. Загрузка ResNet critic (уже обучен)

In [ ]:
from torchvision.models import resnet18

class ImageEncoder(nn.Module):
    def __init__(self, out_dim=512):
        super().__init__()
        self.backbone = resnet18(weights=None)
        self.backbone.fc = nn.Identity()
        self.out_dim = 512
    def forward(self, img):
        return self.backbone(img)


class CriticHead(nn.Module):
    def __init__(self, obs_dim, action_dim=7, hidden=512, action_emb_dim=128):
        super().__init__()
        self.action_emb = nn.Sequential(
            nn.Linear(action_dim, action_emb_dim), nn.LayerNorm(action_emb_dim), nn.ReLU(),
            nn.Linear(action_emb_dim, action_emb_dim), nn.LayerNorm(action_emb_dim), nn.ReLU(),
        )
        self.l1 = nn.Sequential(nn.Linear(obs_dim+action_emb_dim, hidden), nn.LayerNorm(hidden), nn.ReLU())
        self.l2 = nn.Sequential(nn.Linear(hidden+action_emb_dim, hidden), nn.LayerNorm(hidden), nn.ReLU())
        self.l3 = nn.Sequential(nn.Linear(hidden+action_emb_dim, hidden), nn.LayerNorm(hidden), nn.ReLU())
        self.l4 = nn.Linear(hidden+action_emb_dim, 1)
    def forward(self, obs, a):
        a_emb = self.action_emb(a)
        x = self.l1(torch.cat([obs, a_emb], -1))
        x = self.l2(torch.cat([x, a_emb], -1))
        x = self.l3(torch.cat([x, a_emb], -1))
        return self.l4(torch.cat([x, a_emb], -1)).squeeze(-1)


class CriticEnsemble(nn.Module):
    def __init__(self, state_dim=8, action_dim=7, hidden=512, state_hidden=64):
        super().__init__()
        self.enc1 = ImageEncoder()
        self.enc2 = ImageEncoder()
        self.state_mlp = nn.Sequential(
            nn.Linear(state_dim, state_hidden), nn.LayerNorm(state_hidden), nn.ReLU(),
            nn.Linear(state_hidden, state_hidden),
        )
        self.obs_dim = 512 + 512 + state_hidden
        self.obs_norm = nn.LayerNorm(self.obs_dim)
        self.q1 = CriticHead(self.obs_dim, action_dim, hidden)
        self.q2 = CriticHead(self.obs_dim, action_dim, hidden)
    def encode(self, img1, img2, state):
        e1 = self.enc1(img1)
        e2 = self.enc2(img2)
        s  = self.state_mlp(state)
        obs = torch.cat([e1, e2, s], dim=-1)
        return self.obs_norm(obs)
    def forward(self, img1, img2, state, action):
        obs = self.encode(img1, img2, state)
        return self.q1(obs, action), self.q2(obs, action)
    def min_q(self, img1, img2, state, action):
        q1, q2 = self.forward(img1, img2, state, action)
        return torch.minimum(q1, q2)


class VNetwork(nn.Module):
    def __init__(self, obs_dim, hidden=512):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.LayerNorm(hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.LayerNorm(hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.LayerNorm(hidden), nn.ReLU(),
            nn.Linear(hidden, 1),
        )
    def forward(self, obs):
        return self.net(obs).squeeze(-1)


critic = CriticEnsemble().to(device)
v_net = VNetwork(critic.obs_dim).to(device)

state = torch.load(CRITIC_PATH, map_location=device, weights_only=False)
critic.load_state_dict(state["critic"])
v_net.load_state_dict(state["v_net"])
state_mean = state["state_mean"]
state_std  = state["state_std"]
critic.eval()
v_net.eval()
for p in critic.parameters(): p.requires_grad_(False)
for p in v_net.parameters(): p.requires_grad_(False)

print(f"Critic + V loaded from {CRITIC_PATH}")
print(f"  Critic params: {sum(p.numel() for p in critic.parameters())/1e6:.1f}M (frozen)")
print(f"  V params: {sum(p.numel() for p in v_net.parameters())/1e6:.1f}M (frozen)")


## 4. Загрузка cached images для PA-RL training

Используем тот же кэш 256×256 что для критика. На GPU держать всё дорого, но 50k frames × 2 cameras × 256×256 uint8 = ~20 GB всё ещё влезает на 48 GB GPU.

In [ ]:
CRITIC_IMG_SIZE = 128  # было 84: соответствует размеру в critic_resnet.ipynb
RAW_IMG_SIZE    = 256
IMG_CACHE_PATH  = "/workspace/data/img_cache_critic_256.npz"

print(f"Загружаем image cache из {IMG_CACHE_PATH}…")
data = np.load(IMG_CACHE_PATH)
img1_cache = data["img1"]
img2_cache = data["img2"]
state_cache = data["state"].astype(np.float32)
action_cache = data["action"].astype(np.float32)
episode_cache = data["episode"].astype(np.int64)
frame_cache = data["frame"].astype(np.int64)
print(f"  loaded: {img1_cache.shape} ({img1_cache.nbytes/1e9:.1f} GB)")

# Нормализуем state
# state_mean / state_std сохранены как numpy (без .cpu()/.numpy())
state_mean_np = state_mean if isinstance(state_mean, np.ndarray) else state_mean.cpu().numpy()
state_std_np  = state_std  if isinstance(state_std,  np.ndarray) else state_std.cpu().numpy()
state_norm_cache = (state_cache - state_mean_np) / state_std_np

# Sort
sort_order = np.lexsort((frame_cache, episode_cache))
img1_sorted = img1_cache[sort_order]
img2_sorted = img2_cache[sort_order]
state_norm_sorted = state_norm_cache[sort_order]
action_sorted = action_cache[sort_order]
del img1_cache, img2_cache  # освобождаем

# На GPU как uint8 (20 GB)
print("Перенос image cache на GPU…")
img1_gpu = torch.from_numpy(img1_sorted).to(device)
img2_gpu = torch.from_numpy(img2_sorted).to(device)
state_gpu = torch.from_numpy(state_norm_sorted).to(device)
action_gpu = torch.from_numpy(action_sorted).to(device)
print(f"  GPU memory used: {torch.cuda.memory_allocated()/1e9:.1f} GB")
N = img1_gpu.shape[0]
print(f"  N transitions: {N}")


def downscale_only(img, target_size=CRITIC_IMG_SIZE):
    if target_size != img.shape[-1]:
        return F.interpolate(img, size=target_size, mode='bilinear', align_corners=False)
    return img


# ── Чтобы дать в distillation REAL chunk action[t..t+chunk_size] из dataset ──
# action_gpu отсортирован по (episode, frame). Для каждой позиции idx[i] нам нужен
# chunk action[idx[i]+0, idx[i]+1, ..., idx[i]+chunk_size-1].
# Важно: если эпизод закончится раньше — копируем последний действительный action.
chunk_size = policy.config.chunk_size
print(f"Готовим action chunks длиной {chunk_size} для distillation…")

# Boundaries эпизодов — где ep[i+1] != ep[i] (на отсортированном data)
episode_sorted = episode_cache[sort_order]
is_last_in_ep = np.concatenate((episode_sorted[1:] != episode_sorted[:-1], [True]))
is_last_t = torch.from_numpy(is_last_in_ep.astype(np.bool_)).to(device)

# Готовим idx → idx+0, idx+1, ..., idx+chunk_size-1 with episode-aware clipping
# Самый простой подход: для каждого frame готовим chunk_idxs [B, chunk_size]
# = [idx, idx+1, ..., idx+chunk_size-1], но clip if пересекли эпизод.
# Делаем это лениво в sample.

def sample_batch_for_parl(batch_size):
    """Семплирует батч для PA-RL + REAL action chunks из dataset для distillation."""
    idx = torch.randint(0, N - chunk_size, (batch_size,), device=device)  # -chunk_size чтобы не выйти
    
    # Для critic — downscaled images + state
    img1_critic = img1_gpu[idx].permute(0, 3, 1, 2).float() / 255.0
    img2_critic = img2_gpu[idx].permute(0, 3, 1, 2).float() / 255.0
    img1_critic = downscale_only(img1_critic)
    img2_critic = downscale_only(img2_critic)
    state_critic = state_gpu[idx]
    
    # Для policy — full images
    img1_policy = img1_gpu[idx].permute(0, 3, 1, 2).float() / 255.0
    img2_policy = img2_gpu[idx].permute(0, 3, 1, 2).float() / 255.0
    
    state_mean_t = torch.tensor(state_mean_np, device=device, dtype=torch.float32)
    state_std_t  = torch.tensor(state_std_np, device=device, dtype=torch.float32)
    state_raw = state_gpu[idx] * state_std_t + state_mean_t
    
    # Demo action — текущий action[idx]
    action_demo = action_gpu[idx]
    
    # ── REAL action chunks: [B, chunk_size, 7] ──
    # action[idx+t] для t=0..chunk_size-1, с clip по эпизоду
    chunk_idxs = idx.unsqueeze(1) + torch.arange(chunk_size, device=device).unsqueeze(0)  # [B, chunk]
    # Найдём где пересекли эпизод (is_last_t in chunk_idxs[:, :t]) — после последнего
    # действительного frame'а в эпизоде, копируем последний action.
    # Проще: для каждого batch element найти первый t где is_last_t[chunk_idxs[b, t]]=True и далее заклемпить.
    is_last_at = is_last_t[chunk_idxs]  # [B, chunk]
    # cumulative — после первого True все True
    crossed = is_last_at.cumsum(dim=1).clamp(max=1).bool()  # [B, chunk]
    # last_valid index per batch — где crossed впервые становится True
    # Maintain idx of last valid frame: до 1st True берём свой position, дальше — позицию того True
    first_end = crossed.float().argmax(dim=1)  # [B] — index первого True (или 0 если нет True)
    # Если crossed[..., 0] == False везде, argmax даст 0 — но мы хотим всё взять
    no_crossing = (~crossed).all(dim=1)
    first_end = torch.where(no_crossing, torch.full_like(first_end, chunk_size - 1), first_end)
    
    # final_idxs: для t<=first_end[b] → chunk_idxs[b,t]; для t>first_end[b] → chunk_idxs[b, first_end[b]]
    t_range = torch.arange(chunk_size, device=device).unsqueeze(0).expand(batch_size, -1)
    use_t = torch.minimum(t_range, first_end.unsqueeze(1))  # [B, chunk]
    final_idxs = torch.gather(chunk_idxs, 1, use_t)  # [B, chunk]
    
    action_chunk_real = action_gpu[final_idxs.flatten()].view(batch_size, chunk_size, 7)
    
    return {
        "img1_critic": img1_critic,
        "img2_critic": img2_critic,
        "state_critic": state_critic,
        "img1_policy": img1_policy,
        "img2_policy": img2_policy,
        "state_raw": state_raw,
        "action_demo": action_demo,
        "action_chunk_real": action_chunk_real,  # [B, chunk_size, 7] — настоящие demo chunks
    }


## 5. Sample N actions from base policy через flow-matching

Адаптация SmolVLA flow-matching: для одного prefix делаем N независимых denoise проходов с разными noise → N action chunks.

In [ ]:
@torch.no_grad()
def prepare_policy_inputs(img1_policy, img2_policy, state_raw, batch_size):
    """Готовит inputs ЕДИНЫМ путём через policy.prepare_images / prepare_state.
    
    КЛЮЧЕВОЕ ОТЛИЧИЕ от прошлой версии: вместо ручного `resize_with_pad * 2 - 1`
    пускаем картинки через `policy.prepare_images(raw)` — тот же код, что
    использует distillation forward (policy.model.forward). Это гарантирует, что
    sampling и distill видят BIT-IDENTICAL изображения.
    """
    # State + language через preprocessor
    sp = preprocessor({
        "observation.images.image":       torch.zeros(batch_size, 3, 4, 4),
        "observation.images.wrist_image": torch.zeros(batch_size, 3, 4, 4),
        "observation.state":              state_raw.cpu(),
        "task":                           [TASK_DESCRIPTION] * batch_size,
    })
    state_norm  = sp["observation.state"].to(device)
    lang_tokens = sp["observation.language.tokens"].to(device)
    lang_masks  = sp["observation.language.attention_mask"].to(device)
    
    # Images + state через policy.prepare_* — единый источник истины
    raw = {
        "observation.images.image":            img1_policy,
        "observation.images.image2":           img2_policy,
        "observation.images.wrist_image":      img2_policy,
        "observation.state":                   state_norm,
        "observation.language.tokens":         lang_tokens,
        "observation.language.attention_mask": lang_masks,
    }
    images, img_masks = policy.prepare_images(raw)
    state_p           = policy.prepare_state(raw)
    
    return raw, images, img_masks, state_p, lang_tokens, lang_masks


@torch.no_grad()
def sample_n_actions(images, img_masks, state_p, lang_tokens, lang_masks,
                    n_candidates=None, num_steps_override=None):
    """Семплирует n_candidates ПОЛНЫХ action chunks через flow-matching.
    
    Принимает images/img_masks (output of prepare_images) — те же тензоры,
    которые потом увидит distillation forward. Никаких отдельных image paths.
    """
    if n_candidates is None:
        n_candidates = N_CANDIDATES
    
    B = images[0].shape[0] if isinstance(images, list) else images.shape[0]
    
    # Embed prefix один раз — кэшируем для всех N candidates
    prefix_embs, prefix_pad_masks, prefix_att_masks = policy.model.embed_prefix(
        images, img_masks, lang_tokens, lang_masks, state=state_p
    )
    prefix_att_2d  = make_att_2d_masks(prefix_pad_masks, prefix_att_masks)
    prefix_pos_ids = torch.cumsum(prefix_pad_masks, dim=1) - 1
    _, past_kv = policy.model.vlm_with_expert.forward(
        attention_mask=prefix_att_2d, position_ids=prefix_pos_ids,
        past_key_values=None, inputs_embeds=[prefix_embs, None],
        use_cache=True, fill_kv_cache=True,
    )
    
    actions_shape = (B, policy.config.chunk_size, policy.config.max_action_dim)
    num_steps = num_steps_override if num_steps_override is not None else NUM_DENOISE_STEPS
    dt = -1.0 / num_steps
    
    chunks = []
    for _ in range(n_candidates):
        x_t = policy.model.sample_noise(actions_shape, device)
        for step in range(num_steps):
            t_val = 1.0 + step * dt
            t_tensor = torch.tensor(t_val, device=device).expand(B)
            v_t = policy.model.denoise_step(prefix_pad_masks, past_kv, x_t, t_tensor)
            x_t = x_t + dt * v_t
        chunks.append(x_t[:, :, :7])  # ПОЛНЫЙ chunk [B, chunk_size, 7]
    
    # Stack: [B, N, chunk_size, 7]
    return torch.stack(chunks, dim=1)


## 6. PA-RL шаг: global → local optimization → distillation

Это **точно по Algorithm 1** статьи:
1. Sample N candidates `a_i ~ π(s)`
2. Re-rank по Q, оставить top-m
3. Local refine: M grad steps `a ← a + η·∇_a Q(s,a)`
4. Pick `a*` через categorical sampling из softmax(Q)
5. BC loss `L = MSE(π(s), a*)` через flow-matching forward — **ТОЛЬКО первая позиция**

In [ ]:
def pa_rl_step(batch):
    """Один шаг PA-RL по Algorithm 1 статьи 2412.06685.

    Изменения в этой версии:
    — ЕДИНЫЙ путь подготовки images: prepare_policy_inputs использует
      policy.prepare_images, и тот же `images, img_masks` идут и в sampling,
      и в distillation forward. Никаких различий между train- и eval-распределениями.
    — policy.eval() во время sampling (dropout off), policy.train() для distill.
    — Gripper заморожен в local refine: g[:, 6] = 0; a_new[:, 6] = init.
      Q-функция монотонна по гриппер-измерению, и без freeze он насыщается к ±1.
    """
    img1_critic  = batch["img1_critic"]
    img2_critic  = batch["img2_critic"]
    state_critic = batch["state_critic"]
    img1_policy  = batch["img1_policy"]
    img2_policy  = batch["img2_policy"]
    state_raw    = batch["state_raw"]
    action_demo  = batch["action_demo"]
    B = img1_critic.shape[0]
    chunk_size = policy.config.chunk_size

    # ── 1. ЕДИНЫЙ путь подготовки inputs ──
    raw, images, img_masks, state_p, lang_tokens, lang_masks = \
        prepare_policy_inputs(img1_policy, img2_policy, state_raw, B)

    # ── 2. Sample N полных chunk-ов в EVAL-режиме (dropout off) ──
    policy.eval()
    candidates_full = sample_n_actions(
        images, img_masks, state_p, lang_tokens, lang_masks,
        n_candidates=N_CANDIDATES,
    )  # [B, N, chunk_size, 7]
    policy.train()

    # ── 3. Encode obs для critic ──
    with torch.no_grad():
        obs_critic = critic.encode(img1_critic, img2_critic, state_critic)

    # ── 4. Global rerank по Q первого действия (Eq 4.1) ──
    with torch.no_grad():
        c_first = candidates_full[:, :, 0, :].contiguous()
        obs_exp = obs_critic.unsqueeze(1).expand(-1, N_CANDIDATES, -1)
        q_all = critic.q1(
            obs_exp.reshape(B * N_CANDIDATES, -1),
            c_first.reshape(B * N_CANDIDATES, 7),
        ).reshape(B, N_CANDIDATES)
        topm_q, topm_idx = q_all.topk(M_GLOBAL, dim=1)
        b_idx = torch.arange(B, device=device).unsqueeze(1).expand(-1, M_GLOBAL)
        topm_chunks = candidates_full[b_idx, topm_idx]
        topm_first  = topm_chunks[:, :, 0, :].contiguous()
        q_best_mean = q_all.max(dim=1).values.mean().item()

    # ── 5. Local refine: top-m ∪ {a_data}, gripper FROZEN ──
    K_init = M_GLOBAL + 1
    a_init = torch.cat([topm_first, action_demo.unsqueeze(1)], dim=1)
    a_flat = a_init.reshape(B * K_init, 7)
    obs_K  = obs_critic.unsqueeze(1).expand(-1, K_init, -1).reshape(B * K_init, -1)

    # ★ Сохраняем gripper-измерение — НЕ оптимизируется градиентом
    gripper_init = a_flat[:, 6:7].clone()  # [B*K, 1]

    a      = a_flat.detach().clone()
    sticky = torch.ones(B * K_init, dtype=torch.bool, device=device)
    for _ in range(N_REFINEMENT_STEPS):
        a_req = a.detach().requires_grad_(True)
        q     = critic.q1(obs_K, a_req)
        g     = torch.autograd.grad(q.sum(), a_req)[0]
        g[:, 6] = 0.0                                                   # ★ нулевой grad по gripper
        a_new = (a_req + ETA_LOCAL * g).clamp(-1, 1).detach()
        a_new[:, 6] = gripper_init.squeeze(-1)                          # ★ восстановить gripper
        with torch.no_grad():
            q_new   = critic.q1(obs_K, a_new)
            improved = (q_new > q.detach())
            sticky   = sticky & improved
            a        = torch.where(sticky.unsqueeze(-1), a_new, a)

    refined = a.detach().reshape(B, K_init, 7)

    # ── 6. Eq 4.3 — categorical pick ──
    with torch.no_grad():
        all_q = critic.q1(obs_K, refined.reshape(B * K_init, 7)).reshape(B, K_init)
        q_top_mean = all_q.max(dim=1).values.mean().item()
        q_std = all_q.std(dim=1).mean().item()
        if q_std < 0.1:
            best_idx = all_q.argmax(dim=1)
            a_first  = refined[torch.arange(B), best_idx]
        else:
            weights = torch.softmax(all_q, dim=1)
            sampled = torch.multinomial(weights, num_samples=1).squeeze(-1)
            a_first = refined[torch.arange(B), sampled]                  # [B, 7]

    # ── 7. Target chunk = [a*, хвост из top-1 sampled chunk-а] ──
    a_rest        = topm_chunks[:, 0, 1:, :].detach()
    target_chunk  = torch.cat([a_first.unsqueeze(1), a_rest], dim=1)
    target_padded = pad_vector(target_chunk, policy.config.max_action_dim)

    # ── 8. BC distillation forward — те же images/state, что в sampling ──
    losses  = policy.model.forward(images, img_masks, lang_tokens, lang_masks, state_p, target_padded)
    bc_loss = losses[:, 0, :7].mean()  # position 0 only

    lora_l2 = 0.0
    for n, p in policy.named_parameters():
        if p.requires_grad and "lora_B" in n:
            lora_l2 = lora_l2 + p.pow(2).sum()
    total_loss = bc_loss + LORA_L2_COEF * lora_l2

    policy_optimizer.zero_grad()
    total_loss.backward()
    grad_norm = torch.nn.utils.clip_grad_norm_(
        [p for p in policy.parameters() if p.requires_grad], max_norm=1.0
    )
    policy_optimizer.step()

    with torch.no_grad():
        q_demo_mean = critic.q1(obs_critic, action_demo).mean().item()
        q_pol_mean  = critic.q1(obs_critic, a_first).mean().item()

    return {
        "bc_loss":   bc_loss.item(),
        "lora_l2":   (lora_l2.item() if torch.is_tensor(lora_l2) else lora_l2),
        "q_demo":    q_demo_mean,
        "q_pol":     q_pol_mean,
        "q_top":     q_top_mean,
        "q_best":    q_best_mean,
        "q_std":     q_std,
        "grad_norm": grad_norm.item(),
    }


## 7. PA-RL training loop

In [ ]:
# ── Progressive PA-RL training с эвалом и сохранением best checkpoint ──
# Sweet spot: SR растёт первые 20-60 шагов, потом обваливается. Будем сохранять best.

policy.train()

# ── Param-groups: WD=0.05 на LoRA, ~0 на остальное (action_out_proj) ──
# Большой WD на LoRA-A/B даёт дополнительную anti-drift регуляризацию
# поверх LORA_L2_COEF на B-матрицу.
lora_params  = [p for n, p in policy.named_parameters() if "lora_" in n and p.requires_grad]
other_params = [p for n, p in policy.named_parameters() if "lora_" not in n and p.requires_grad]
policy_optimizer = optim.AdamW(
    [{"params": lora_params,  "weight_decay": 0.05},
     {"params": other_params, "weight_decay": 1e-6}],
    lr=POLICY_LR,
)
print(f"Optimizer: AdamW lr={POLICY_LR}, "
      f"LoRA params={sum(p.numel() for p in lora_params)/1e6:.2f}M (WD=0.05), "
      f"other={sum(p.numel() for p in other_params)/1e6:.2f}M (WD=1e-6)")

LORA_L2_COEF = 0.001  # было 0.01 — после фикса target_chunk LoRA меньше дрейфует
print(f"LoRA L2 regularization coef: {LORA_L2_COEF}")

# ── env init для эвала ──
import builtins
builtins.input = lambda _: "n"
os.environ["LIBERO_DATA_PATH"] = "/workspace/libero_data"
os.makedirs("/workspace/libero_data", exist_ok=True)

from libero.libero import get_libero_path, benchmark
from libero.libero.envs import OffScreenRenderEnv

benchmark_dict = benchmark.get_benchmark_dict()
task_suite = benchmark_dict[TASK_SUITE_NAME]()
task = task_suite.get_task(TASK_ID)
task_description_libero = task.language
task_bddl_file = os.path.join(get_libero_path("bddl_files"), task.problem_folder, task.bddl_file)

env = OffScreenRenderEnv(bddl_file_name=task_bddl_file,
                         camera_heights=ENV_IMAGE_SIZE, camera_widths=ENV_IMAGE_SIZE)
env.seed(SEED)
init_states = task_suite.get_task_init_states(TASK_ID)


def quat2axisangle(quat):
    quat = quat.astype(np.float32).copy()
    quat[3] = np.clip(quat[3], -1.0, 1.0)
    den = np.sqrt(max(1e-12, 1.0 - quat[3]**2))
    if np.isclose(den, 0.0): return np.zeros(3, dtype=np.float32)
    angle = 2.0 * math.acos(float(quat[3]))
    return ((quat[:3] * angle) / den).astype(np.float32)

def rotate_180(im): return np.ascontiguousarray(im[::-1, ::-1])

def libero_obs_to_lerobot(raw_obs, task_text):
    a_img = rotate_180(raw_obs["agentview_image"])
    w_img = rotate_180(raw_obs["robot0_eye_in_hand_image"])
    eef_pos = raw_obs["robot0_eef_pos"].astype(np.float32)
    eef_quat = raw_obs["robot0_eef_quat"].astype(np.float32)
    grip = raw_obs["robot0_gripper_qpos"].astype(np.float32)
    state = np.concatenate([eef_pos, quat2axisangle(eef_quat), grip], 0).astype(np.float32)
    return {
        "observation.images.image":  torch.from_numpy(a_img).permute(2,0,1).unsqueeze(0).float()/255.0,
        "observation.images.image2": torch.from_numpy(w_img).permute(2,0,1).unsqueeze(0).float()/255.0,
        "observation.state":         torch.from_numpy(state).unsqueeze(0).float(),
        "task":                      [task_text],
    }


@torch.no_grad()
def predict_action(raw_obs, task_text):
    obs = libero_obs_to_lerobot(raw_obs, task_text)
    obs = preprocessor(obs)
    a = policy.select_action(obs)
    a = postprocessor(a)
    return a.squeeze(0).detach().cpu().numpy().astype(np.float32)


def eval_libero(label=""):
    """Эвал на 13 init_states + per-dim action stats."""
    policy.eval()
    n_success = 0
    all_actions = []
    t_eval_start = time.time()
    for state_id in INIT_STATES_IDS:
        try:
            policy.reset()
            env.reset()
            raw_obs = env.set_init_state(init_states[state_id])
            success = False
            ep_actions = []
            for t in range(MAX_STEPS + NUM_STABILIZATION_STEPS):
                if t < NUM_STABILIZATION_STEPS:
                    action = [0.0]*6 + [-1.0]
                else:
                    a = predict_action(raw_obs, task_description_libero)
                    action = a.tolist()
                    ep_actions.append(a)
                raw_obs, _, done, _ = env.step(action)
                if done:
                    success = True
                    break
            if success: n_success += 1
            all_actions.extend(ep_actions)
            elapsed = time.time() - t_eval_start
            print(f"    state {state_id}: {'OK' if success else '..'} ({t+1} steps, {elapsed:.0f}s)", flush=True)
        except Exception as e:
            print(f"    state {state_id}: ERROR - {type(e).__name__}: {str(e)[:80]}", flush=True)
            continue
    
    all_actions = np.array(all_actions) if all_actions else np.zeros((1, 7))
    sr = n_success / len(INIT_STATES_IDS)
    mean_abs = np.abs(all_actions).mean(axis=0) if len(all_actions) else np.zeros(7)
    print(f"  [{label}] SR: {n_success}/13 = {sr:.0%}, action mean abs: {mean_abs.round(3)}", flush=True)
    policy.train()
    return n_success, sr, mean_abs


def save_checkpoint(path):
    """Сохраняет policy + processor files."""
    os.makedirs(path, exist_ok=True)
    policy.save_pretrained(path)
    import shutil
    src_proc = MODEL_ID.replace("/", "--")
    hf_cache = os.path.expanduser(f"~/.cache/huggingface/hub/models--{src_proc}/snapshots")
    if os.path.exists(hf_cache):
        snapshot_dirs = [d for d in os.listdir(hf_cache) if not d.startswith(".")]
        if snapshot_dirs:
            proc_src = os.path.join(hf_cache, snapshot_dirs[0])
            for fname in os.listdir(proc_src):
                if fname.startswith("policy_"):
                    shutil.copy2(os.path.join(proc_src, fname), os.path.join(path, fname))


# ── Progressive training loop ──
print("="*70)
print(f"PROGRESSIVE PA-RL: эвалы на шагах {EVAL_AT_STEPS}, max {PARL_STEPS}")
print("="*70)

eval_results = {}
best_n_success = -1
best_step = -1
SAVE_BEST_PATH = SAVE_VLA_PATH + "_best"

# Eval baseline (step 0)
print(f"\n--- step 0 (baseline) ---")
n_succ, sr, mean_abs = eval_libero(label="step 0")
eval_results[0] = (n_succ, sr, mean_abs)
if n_succ > best_n_success:
    best_n_success = n_succ
    best_step = 0
    print(f"  ✓ BEST so far: {n_succ}/13 (saving)")
    save_checkpoint(SAVE_BEST_PATH)

# Train + eval cycle
prev_step = 0
recent_metrics = {"bc": [], "l2": [], "q_gap": []}
t0_total = time.time()

for next_eval_step in EVAL_AT_STEPS:
    if next_eval_step == 0:
        continue
    n_steps_to_train = next_eval_step - prev_step
    print(f"\n--- training {prev_step}->{next_eval_step} ({n_steps_to_train} steps) ---")
    
    for step in range(n_steps_to_train):
        batch = sample_batch_for_parl(BATCH_SIZE)
        m = pa_rl_step(batch)
        recent_metrics["bc"].append(m["bc_loss"])
        recent_metrics["l2"].append(m["lora_l2"])
        recent_metrics["q_gap"].append(m["q_pol"] - m["q_demo"])
        for k in recent_metrics:
            if len(recent_metrics[k]) > 10:
                recent_metrics[k].pop(0)
        
        # Лог каждые LOG_EVERY
        if (step + 1) % LOG_EVERY == 0 or step == 0:
            absolute_step = prev_step + step + 1
            elapsed = time.time() - t0_total
            print(f"  step {absolute_step:3d} | bc={m['bc_loss']:.3f} | l2={m['lora_l2']:.2f} | "
                  f"q_demo={m['q_demo']:.1f} q_pol={m['q_pol']:.1f} (gap={m['q_pol']-m['q_demo']:+.2f}) | "
                  f"grad={m['grad_norm']:.2f} | t={elapsed/60:.1f}m", flush=True)
    
    avg_bc = sum(recent_metrics["bc"]) / max(len(recent_metrics["bc"]), 1)
    avg_l2 = sum(recent_metrics["l2"]) / max(len(recent_metrics["l2"]), 1)
    avg_qgap = sum(recent_metrics["q_gap"]) / max(len(recent_metrics["q_gap"]), 1)
    elapsed = time.time() - t0_total
    print(f"  trained. avg bc={avg_bc:.3f} l2={avg_l2:.2f} q_gap={avg_qgap:+.2f}, elapsed={elapsed/60:.1f}min")
    
    n_succ, sr, mean_abs = eval_libero(label=f"step {next_eval_step}")
    eval_results[next_eval_step] = (n_succ, sr, mean_abs)
    
    # Сохраняем best checkpoint
    if n_succ > best_n_success:
        best_n_success = n_succ
        best_step = next_eval_step
        print(f"  ✓ NEW BEST: {n_succ}/13 at step {best_step} (saving)")
        save_checkpoint(SAVE_BEST_PATH)
    elif n_succ < best_n_success - 2:
        print(f"  ⚠ Падение SR на 2+ от best ({best_n_success}/13). Останавливаемся.")
    
    prev_step = next_eval_step

# === Итоги ===
print("\n" + "="*70)
print("ИТОГИ")
print("="*70)
print(f"\n{'step':<8} {'SR':<14} {'x':<8} {'y':<8} {'z':<8} {'gripper':<10}")
for step in sorted(eval_results.keys()):
    n_succ, sr, mean_abs = eval_results[step]
    is_best = " <-- BEST" if step == best_step else ""
    print(f"{step:<8} {n_succ}/13 ({sr:.0%})   "
          f"{mean_abs[0]:<8.3f} {mean_abs[1]:<8.3f} {mean_abs[2]:<8.3f} {mean_abs[6]:<10.3f}{is_best}")

print(f"\nBest checkpoint: step {best_step}, SR = {best_n_success}/13")
print(f"Сохранён в: {SAVE_BEST_PATH}")

baseline_succ = eval_results[0][0]
if best_n_success > baseline_succ:
    print(f"\n✓ PA-RL улучшил SR на +{best_n_success - baseline_succ} эпизод(а) vs baseline ({baseline_succ}/13).")
elif best_n_success == baseline_succ:
    print(f"\n○ PA-RL = baseline ({baseline_succ}/13). Никакого вреда но и пользы.")
else:
    print(f"\n⚠ PA-RL хуже baseline ({best_n_success} vs {baseline_succ}). Оставляем baseline.")



## 8. LIBERO env + эвал на 13 init_states

In [ ]:
# (disabled — eval теперь делается inline в training cell)
pass


## 9. Вердикт

In [ ]:
# (disabled — eval теперь делается inline в training cell)
pass
